In [1]:
import torch
from torch.utils.data import DataLoader
import torchvision
from torchvision.datasets import ImageFolder
from torchvision.transforms import Compose, ToTensor, Normalize, RandomHorizontalFlip, RandomResizedCrop, RandomRotation
from tqdm import tqdm
from torchmetrics.functional import accuracy

from torch.utils.tensorboard import SummaryWriter
writer = SummaryWriter('runs/catsdogs/experiment_5')
torch.backends.cudnn.benchmark = True

In [ ]:
# Resize all images in folder to 224x224    
import cv2
import os
import numpy as np
from PIL import Image

os.makedirs('PetResizeSmall/Cat', exist_ok=True)
os.makedirs('PetResizeSmall/Dog', exist_ok=True)

def resize_images_in_folder(folder_path, size, new_folder_path):
    for filename in os.listdir(folder_path):
        #print(filename)
        if filename.endswith('.jpg'):
            img = cv2.imread(os.path.join(folder_path, filename))
            if img is None:
                continue
            #print(img.shape)
            img = cv2.resize(img, size, interpolation=cv2.INTER_LINEAR)
            cv2.imwrite(os.path.join(new_folder_path, filename), img)

#resize_images_in_folder('PetImages/Cat', (224, 224), 'PetResizeSmall/Cat')
#resize_images_in_folder('PetImages/Dog', (224, 224), 'PetResizeSmall/Dog')

In [6]:
train_dataset = ImageFolder(root='PetResizeSmall/train',
                            transform=Compose([ToTensor() , Normalize((0.5, 0.5, 0.5), (0.5, 0.5, 0.5)), RandomHorizontalFlip(), RandomResizedCrop(224), RandomRotation(30)]))  
val_dataset = ImageFolder(root='PetResizeSmall/val',
                            transform=Compose([ToTensor() , Normalize((0.5, 0.5, 0.5), (0.5, 0.5, 0.5))]))

FileNotFoundError: [WinError 3] The system cannot find the path specified: 'PetResizeSmall/train'

In [4]:
train_loader = DataLoader(train_dataset, batch_size=6, shuffle=True, num_workers=4,pin_memory=True, persistent_workers=True, prefetch_factor=6)
val_loader = DataLoader(val_dataset, batch_size=8, shuffle=False)

In [5]:
model = torchvision.models.vgg16(pretrained=True)
for param in model.parameters():
    param.requires_grad = False
model.classifier[6] = torch.nn.Linear(4096, 1)
model = model.cuda()

c:\Users\lucas\miniconda3\envs\torch-gpu\lib\site-packages\torchvision\models\_utils.py:208: UserWarning: The parameter 'pretrained' is deprecated since 0.13 and may be removed in the future, please use 'weights' instead.
  warnings.warn(
c:\Users\lucas\miniconda3\envs\torch-gpu\lib\site-packages\torchvision\models\_utils.py:223: UserWarning: Arguments other than a weight enum or `None` for 'weights' are deprecated since 0.13 and may be removed in the future. The current behavior is equivalent to passing `weights=VGG16_Weights.IMAGENET1K_V1`. You can also use `weights=VGG16_Weights.DEFAULT` to get the most up-to-date weights.
  warnings.warn(msg)


In [6]:
def val_step(epoch, model:torch.nn.Module, loss_fn:torch.nn.Module, dataloader:torch.utils.data.DataLoader, device:torch.device):
    test_loss, test_acc = 0, 0 
    model.eval()
    with torch.inference_mode():
        for X, y in dataloader:
            X, y = X.to(device), y.to(device)
            y = y.unsqueeze(1)

            # 1. Forward pass
            test_pred = model(X)
        
            # 2. Loss
            test_loss += loss_fn(test_pred, y.float()).item() # accumulatively add up the loss per epoch

            # 3. Computa a acuracia
            test_acc += accuracy(target=y,
                                 preds=torch.sigmoid(test_pred),
                                 task='binary')
        
        test_loss /= len(dataloader)
        test_acc /= len(dataloader)
        writer.add_scalar('Loss/test', 
                          test_loss,
                          epoch)
        writer.add_scalar('Accuracy/test',
                            test_acc,
                            epoch)
        return test_loss, test_acc

In [7]:
def train_step(epoch, model, dataloader, loss_fn, optimizer, device: torch.device):
    train_loss = 0
    # Faz loop em todos os dados de treino
    model.train()
    with torch.enable_grad():
        for i, (X, y) in enumerate(dataloader):
            X, y = X.to(device), y.to(device)
            y = y.unsqueeze(1)
            model.train() 
            # 1. Forward pass
            y_pred = model(X)

            # 2. Calcula loss por batch
            loss = loss_fn(y_pred, y.float())
            train_loss += loss.item()
            writer.add_scalar('Loss/train',
                                loss.item(),
                                epoch * len(dataloader) + i)
            writer.add_scalar('Accuracy/train',
                                accuracy(torch.sigmoid(y_pred), 
                                        y,
                                        task='binary'),
                                epoch * len(dataloader) + i)

            dataloader.set_postfix({'loss': loss.item()})
            # 3. zera gradientes anteriores
            optimizer.zero_grad()

            # 4. Backward Pass
            loss.backward()

            # 5. Otimizacao
            optimizer.step()


    train_loss /= len(dataloader)
    return train_loss


In [8]:
torch.manual_seed(42)
EPOCHS = 4
LEARNING_RATE = 0.0001
loss_fn = torch.nn.BCEWithLogitsLoss()
optimizer = torch.optim.Adam(model.parameters(), lr=LEARNING_RATE)
device = 'cuda' if torch.cuda.is_available() else 'cpu'
for epoch in range(EPOCHS):
    with tqdm(train_loader, desc=f'{epoch=}', unit='batch') as tqdm_epoch:
        train_loss = train_step(epoch, model, tqdm_epoch, loss_fn, optimizer, device)
        val_loss, val_acc = val_step(epoch, model, loss_fn, val_loader, device)
        print(f'{epoch=}, {train_loss=}, {val_loss=}, {val_acc=}')
        

epoch=0: 100%|██████████| 333/333 [03:11<00:00,  1.74batch/s, loss=0.314] 


epoch=0, train_loss=0.4446699848851642, val_loss=0.14229153359637542, val_acc=tensor(0.9583, device='cuda:0')


epoch=1: 100%|██████████| 333/333 [02:24<00:00,  2.30batch/s, loss=0.0467]


epoch=1, train_loss=0.3159597298667372, val_loss=0.09894113640720938, val_acc=tensor(0.9779, device='cuda:0')


epoch=2: 100%|██████████| 333/333 [02:20<00:00,  2.36batch/s, loss=0.168] 


epoch=2, train_loss=0.27987516370487286, val_loss=0.08350559333156721, val_acc=tensor(0.9755, device='cuda:0')


epoch=3: 100%|██████████| 333/333 [02:18<00:00,  2.41batch/s, loss=0.507] 


epoch=3, train_loss=0.26888062533091855, val_loss=0.07760596068977725, val_acc=tensor(0.9755, device='cuda:0')


In [10]:
torch.save(model, 'model.pth')

In [10]:
import PIL
from PIL import Image
img = Image.open('dog.jpg')
transforms = Compose([ToTensor() , Normalize((0.5, 0.5, 0.5), (0.5, 0.5, 0.5))])
img = transforms(img)
img = img.unsqueeze(0)
img = img.cuda()
model.eval()
with torch.inference_mode():
    pred = torch.sigmoid(model(img))
    print(pred)
    print('dog' if pred > 0.5 else 'cat')

tensor([[0.7556]], device='cuda:0')
dog
